# Essay 2 Notebook: Recommender Systems (Lab-Aligned)

This notebook explains the recommender-system perspective of our lab:

- recommendation paradigm used (content-based)
- retrieval flow (query -> embedding -> vector search)
- quality inspection and simple evaluation
- strengths, limitations, and future hybrid path

It is designed as practical support for **Essay 2 (Recommender Systems)**.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_PATH = ROOT / "data/processed/kdramas_clean.jsonl"
EMB_PATH = ROOT / "data/processed/kdramas_embeddings.jsonl"


def read_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows

clean_df = pd.DataFrame(read_jsonl(CLEAN_PATH))
emb_rows = read_jsonl(EMB_PATH)
emb_df = pd.DataFrame({
    "imdb_id": [r["imdb_id"] for r in emb_rows],
    "vector": [np.array(r["vector"], dtype=np.float32) for r in emb_rows],
})

df = clean_df.merge(emb_df, on="imdb_id", how="inner")
print("rows in recommender corpus:", len(df))
df[["imdb_id", "primary_title", "genres", "rating", "votes"]].head()

rows in recommender corpus: 28


,imdb_id,primary_title,genres,rating,votes
0,tt0086817,Transformers,"[Animation, Action, Adventure, Family, Sci-Fi]",8.0,25334
1,tt0077008,Fantasy Island,"[Adventure, Family, Fantasy, Romance]",6.6,10004
2,tt0088510,Star Wars: Droids,"[Animation, Action, Adventure, Comedy, Family,...",5.8,2582
3,tt0081933,The Smurfs,"[Animation, Adventure, Comedy, Family, Fantasy]",7.2,23511
4,tt0088528,Gummi Bears,"[Animation, Adventure, Family, Fantasy]",7.5,13005


In [2]:
# Content-based retrieval helper (item-to-item)
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def recommend_by_title(title: str, k: int = 5):
    idx = df.index[df["primary_title"].str.lower() == title.lower()]
    if len(idx) == 0:
        raise ValueError(f"title not found: {title}")

    i = int(idx[0])
    qv = df.loc[i, "vector"]
    recs = []
    for j, row in df.iterrows():
        if j == i:
            continue
        recs.append({
            "imdb_id": row["imdb_id"],
            "title": row["primary_title"],
            "genres": ", ".join(row.get("genres", [])) if isinstance(row.get("genres", []), list) else "",
            "score": cosine_sim(qv, row["vector"]),
        })

    rec_df = pd.DataFrame(recs).sort_values("score", ascending=False).head(k)
    return rec_df.reset_index(drop=True)

recommend_by_title("Rugrats", k=5)

,imdb_id,title,genres,score
0,tt0167743,The Wild Thornberrys,"Animation, Adventure, Comedy, Family, Fantasy",0.454803
1,tt0131664,The Angry Beavers,"Animation, Adventure, Comedy, Family",0.445205
2,tt0105941,Animaniacs,"Animation, Short, Adventure, Comedy, Family, M...",0.442254
3,tt0098929,Tiny Toon Adventures,"Animation, Adventure, Comedy, Family, Fantasy,...",0.441514
4,tt0118360,Johnny Bravo,"Animation, Adventure, Comedy, Family",0.403083


In [4]:
# Mini qualitative evaluation used in this project (Essay 2 evidence)
anchors = [
    "Rugrats",
    "X-Men",
    "Transformers",
]

results = {}
for a in anchors:
    try:
        results[a] = recommend_by_title(a, k=5)
    except ValueError:
        results[a] = pd.DataFrame(columns=["imdb_id", "title", "genres", "score"])

for anchor, rec_df in results.items():
    print(f"\n=== Anchor: {anchor} ===")
    display(rec_df)


=== Anchor: Rugrats ===


,imdb_id,title,genres,score
0,tt0167743,The Wild Thornberrys,"Animation, Adventure, Comedy, Family, Fantasy",0.454803
1,tt0131664,The Angry Beavers,"Animation, Adventure, Comedy, Family",0.445205
2,tt0105941,Animaniacs,"Animation, Short, Adventure, Comedy, Family, M...",0.442254
3,tt0098929,Tiny Toon Adventures,"Animation, Adventure, Comedy, Family, Fantasy,...",0.441514
4,tt0118360,Johnny Bravo,"Animation, Adventure, Comedy, Family",0.403083



=== Anchor: X-Men ===


,imdb_id,title,genres,score
0,tt0115378,Superman: The Animated Series,"Animation, Action, Adventure, Family, Sci-Fi",0.497527
1,tt0086817,Transformers,"Animation, Action, Adventure, Family, Sci-Fi",0.497103
2,tt0131613,Teenage Mutant Ninja Turtles,"Animation, Action, Adventure, Comedy, Crime, F...",0.484252
3,tt0105941,Animaniacs,"Animation, Short, Adventure, Comedy, Family, M...",0.446556
4,tt0088631,Thundercats,"Animation, Action, Adventure, Family, Fantasy,...",0.442279



=== Anchor: Transformers ===


,imdb_id,title,genres,score
0,tt0103584,X-Men,"Animation, Action, Adventure, Family, Sci-Fi",0.497103
1,tt0131613,Teenage Mutant Ninja Turtles,"Animation, Action, Adventure, Comedy, Crime, F...",0.470441
2,tt0088510,Star Wars: Droids,"Animation, Action, Adventure, Comedy, Family, ...",0.467001
3,tt0115378,Superman: The Animated Series,"Animation, Action, Adventure, Family, Sci-Fi",0.447277
4,tt0088631,Thundercats,"Animation, Action, Adventure, Family, Fantasy,...",0.444205


## Essay 2 Notes

- **Architecture/technique:** current system is clearly **content-based**
- **Similarity-based recommendation:** cosine score over embedding vectors
- **Real-world discussion:** trade-offs between relevance and diversity
- **Advantages:** no user history required; good cold-start at item side
- **Limitations:** no collaborative signal; quality depends on metadata + embedding model

Future expansion section idea:
- Hybrid recommendation (content + collaborative) once user interaction logs exist.

## Item-to-item vs Query-to-item (Important Interpretation)

In this notebook, the base function `recommend_by_title(title)` is implemented as **item-to-item** recommendation:
- use the anchor item's vector as the query vector,
- exclude the anchor item itself from results.

Therefore, for anchor `Rugrats`, the title `Rugrats` is normally **not shown** in top recommendations.

In the web app/API, recommendation is **query-to-item**:
- free text query -> query embedding -> nearest items in Weaviate.

In query-to-item mode, `Rugrats` can appear as top result for query text `"Rugrats"`.

In [5]:
# Extended item-to-item helper with include_self option
def recommend_by_title(title: str, k: int = 5, include_self: bool = False):
    idx = df.index[df["primary_title"].str.lower() == title.lower()]
    if len(idx) == 0:
        raise ValueError(f"title not found: {title}")

    i = int(idx[0])
    qv = df.loc[i, "vector"]
    recs = []
    for j, row in df.iterrows():
        if (not include_self) and j == i:
            continue
        recs.append({
            "imdb_id": row["imdb_id"],
            "title": row["primary_title"],
            "genres": ", ".join(row.get("genres", [])) if isinstance(row.get("genres", []), list) else "",
            "score": cosine_sim(qv, row["vector"]),
        })

    rec_df = pd.DataFrame(recs).sort_values("score", ascending=False).head(k)
    return rec_df.reset_index(drop=True)

print("Item-to-item (exclude self):")
display(recommend_by_title("Rugrats", k=5, include_self=False))

print("Item-to-item (include self):")
display(recommend_by_title("Rugrats", k=5, include_self=True))

Item-to-item (exclude self):


,imdb_id,title,genres,score
0,tt0167743,The Wild Thornberrys,"Animation, Adventure, Comedy, Family, Fantasy",0.454803
1,tt0131664,The Angry Beavers,"Animation, Adventure, Comedy, Family",0.445205
2,tt0105941,Animaniacs,"Animation, Short, Adventure, Comedy, Family, M...",0.442254
3,tt0098929,Tiny Toon Adventures,"Animation, Adventure, Comedy, Family, Fantasy,...",0.441514
4,tt0118360,Johnny Bravo,"Animation, Adventure, Comedy, Family",0.403083


Item-to-item (include self):


,imdb_id,title,genres,score
0,tt0101188,Rugrats,"Animation, Adventure, Comedy, Family",1.000000
1,tt0167743,The Wild Thornberrys,"Animation, Adventure, Comedy, Family, Fantasy",0.454803
2,tt0131664,The Angry Beavers,"Animation, Adventure, Comedy, Family",0.445205
3,tt0105941,Animaniacs,"Animation, Short, Adventure, Comedy, Family, M...",0.442254
4,tt0098929,Tiny Toon Adventures,"Animation, Adventure, Comedy, Family, Fantasy,...",0.441514


In [6]:
# Query-to-item example (closer to API behavior)
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

def query_to_items(query: str, k: int = 5):
    qv = model.encode([query], normalize_embeddings=True)[0]
    rows = []
    for _, row in df.iterrows():
        rows.append({
            "imdb_id": row["imdb_id"],
            "title": row["primary_title"],
            "genres": ", ".join(row.get("genres", [])) if isinstance(row.get("genres", []), list) else "",
            "score": cosine_sim(qv, row["vector"]),
        })
    return pd.DataFrame(rows).sort_values("score", ascending=False).head(k).reset_index(drop=True)

query_to_items("Rugrats", k=5)

/Volumes/SSD/dev/kdrama-semantic-search-recommendation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9510.89it/s]


,imdb_id,title,genres,score
0,tt0101188,Rugrats,"Animation, Adventure, Comedy, Family",0.630673
1,tt0081933,The Smurfs,"Animation, Adventure, Comedy, Family, Fantasy",0.283277
2,tt0088631,Thundercats,"Animation, Action, Adventure, Family, Fantasy,...",0.209087
3,tt0131664,The Angry Beavers,"Animation, Adventure, Comedy, Family",0.198860
4,tt0131613,Teenage Mutant Ninja Turtles,"Animation, Action, Adventure, Comedy, Crime, F...",0.195194
